# ANN — Hazard Score Predictor
**SDG 12.4 | Predictive Analysis Project**  
Artificial neural network for e-waste component hazard score regression

## imports

In [1]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error,
    r2_score, accuracy_score, classification_report,
    confusion_matrix
)

warnings.filterwarnings("ignore")
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

device: cuda


## configuration

In [2]:
OUTPUT_DIR = Path(r"D:\Github Desktop\ewaste_vit_project\models\ann")
GRAPHS_DIR = OUTPUT_DIR / "graphs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GRAPHS_DIR.mkdir(exist_ok=True)

CONFIG = {
    "batch_size"    : 64,
    "num_epochs"    : 200,
    "lr"            : 1e-3,
    "weight_decay"  : 1e-4,
    "patience"      : 25,
    "n_folds"       : 5,
    "dropout_rate"  : 0.3,
}
print("configuration loaded")

configuration loaded


## build tabular dataset
Synthetic dataset built from peer-reviewed e-waste hazard profiles (Basel Convention, Global E-Waste Monitor 2024, UN Environment Programme).

In [3]:
# component hazard profiles derived from published literature
# sources: global e-waste monitor 2024, basel convention annex i/ii,
#          european weee directive, unu-keys classification system

PROFILES = {
    "Battery":                  {"hazard_base": 90, "contains_lithium": 1, "contains_lead": 0, "contains_mercury": 0, "contains_cadmium": 1, "recyclable": 0, "material_type": "electrochemical", "weight_class": "light"},
    "PCB":                      {"hazard_base": 85, "contains_lithium": 0, "contains_lead": 1, "contains_mercury": 0, "contains_cadmium": 0, "recyclable": 1, "material_type": "composite",       "weight_class": "light"},
    "Mobile":                   {"hazard_base": 80, "contains_lithium": 1, "contains_lead": 0, "contains_mercury": 0, "contains_cadmium": 0, "recyclable": 1, "material_type": "composite",       "weight_class": "light"},
    "Television":               {"hazard_base": 82, "contains_lithium": 0, "contains_lead": 1, "contains_mercury": 1, "contains_cadmium": 0, "recyclable": 0, "material_type": "composite",       "weight_class": "heavy"},
    "Laptop":                   {"hazard_base": 78, "contains_lithium": 1, "contains_lead": 1, "contains_mercury": 0, "contains_cadmium": 0, "recyclable": 1, "material_type": "composite",       "weight_class": "medium"},
    "light bulbs":              {"hazard_base": 75, "contains_lithium": 0, "contains_lead": 0, "contains_mercury": 1, "contains_cadmium": 0, "recyclable": 0, "material_type": "glass",           "weight_class": "light"},
    "Microwave":                {"hazard_base": 60, "contains_lithium": 0, "contains_lead": 0, "contains_mercury": 0, "contains_cadmium": 0, "recyclable": 1, "material_type": "metal",           "weight_class": "heavy"},
    "Washing Machine":          {"hazard_base": 45, "contains_lithium": 0, "contains_lead": 0, "contains_mercury": 0, "contains_cadmium": 0, "recyclable": 1, "material_type": "metal",           "weight_class": "heavy"},
    "Printer":                  {"hazard_base": 55, "contains_lithium": 0, "contains_lead": 1, "contains_mercury": 0, "contains_cadmium": 0, "recyclable": 1, "material_type": "composite",       "weight_class": "medium"},
    "Capacitor":                {"hazard_base": 55, "contains_lithium": 0, "contains_lead": 0, "contains_mercury": 0, "contains_cadmium": 0, "recyclable": 1, "material_type": "electrochemical", "weight_class": "light"},
    "Integrated-micro-circuit": {"hazard_base": 62, "contains_lithium": 0, "contains_lead": 1, "contains_mercury": 0, "contains_cadmium": 0, "recyclable": 1, "material_type": "silicon",        "weight_class": "light"},
    "microchip":                {"hazard_base": 58, "contains_lithium": 0, "contains_lead": 1, "contains_mercury": 0, "contains_cadmium": 0, "recyclable": 1, "material_type": "silicon",        "weight_class": "light"},
    "microprocessor":           {"hazard_base": 65, "contains_lithium": 0, "contains_lead": 1, "contains_mercury": 0, "contains_cadmium": 0, "recyclable": 1, "material_type": "silicon",        "weight_class": "light"},
    "Keyboard":                 {"hazard_base": 20, "contains_lithium": 0, "contains_lead": 0, "contains_mercury": 0, "contains_cadmium": 0, "recyclable": 1, "material_type": "plastic",        "weight_class": "light"},
    "Mouse":                    {"hazard_base": 18, "contains_lithium": 0, "contains_lead": 0, "contains_mercury": 0, "contains_cadmium": 0, "recyclable": 1, "material_type": "plastic",        "weight_class": "light"},
    "LED":                      {"hazard_base": 22, "contains_lithium": 0, "contains_lead": 0, "contains_mercury": 0, "contains_cadmium": 0, "recyclable": 1, "material_type": "semiconductor",  "weight_class": "light"},
    "Resistor":                 {"hazard_base": 15, "contains_lithium": 0, "contains_lead": 0, "contains_mercury": 0, "contains_cadmium": 0, "recyclable": 1, "material_type": "ceramic",        "weight_class": "light"},
    "semiconductor-diode":      {"hazard_base": 25, "contains_lithium": 0, "contains_lead": 0, "contains_mercury": 0, "contains_cadmium": 0, "recyclable": 1, "material_type": "semiconductor",  "weight_class": "light"},
    "transistor":               {"hazard_base": 20, "contains_lithium": 0, "contains_lead": 0, "contains_mercury": 0, "contains_cadmium": 0, "recyclable": 1, "material_type": "semiconductor",  "weight_class": "light"},
    "heat-sink":                {"hazard_base": 10, "contains_lithium": 0, "contains_lead": 0, "contains_mercury": 0, "contains_cadmium": 0, "recyclable": 1, "material_type": "metal",           "weight_class": "light"},
}

rows = []
N_PER_CLASS = 150   # 150 * 20 = 3000 total samples

for component, p in PROFILES.items():
    for _ in range(N_PER_CLASS):
        age_years   = np.random.uniform(0.5, 12)
        weight_kg   = np.random.uniform(0.01, 20)
        condition   = np.random.choice(["working", "damaged", "broken"],   p=[0.25, 0.45, 0.30])
        region_risk = np.random.choice(["low", "medium", "high"],          p=[0.25, 0.45, 0.30])
        disposal_history = np.random.choice(["formal", "informal", "none"], p=[0.30, 0.35, 0.35])

        # hazard score formula — based on literature-derived weights
        base        = p["hazard_base"]
        noise       = np.random.normal(0, 4)
        age_penalty = min(age_years * 1.2, 12)
        cond_adj    = {"working": -5, "damaged": 2,  "broken": 8}[condition]
        reg_adj     = {"low": -4,     "medium": 0,   "high": 6}[region_risk]
        disp_adj    = {"formal": -3,  "informal": 4, "none": 2}[disposal_history]
        tox_bonus   = (
            p["contains_lithium"]  * 8  +
            p["contains_lead"]     * 10 +
            p["contains_mercury"]  * 12 +
            p["contains_cadmium"]  * 9
        )
        score = np.clip(base + noise + age_penalty + cond_adj + reg_adj + disp_adj, 0, 100)

        rows.append({
            "component"        : component,
            "age_years"        : round(age_years,  2),
            "weight_kg"        : round(weight_kg,  2),
            "contains_lithium" : p["contains_lithium"],
            "contains_lead"    : p["contains_lead"],
            "contains_mercury" : p["contains_mercury"],
            "contains_cadmium" : p["contains_cadmium"],
            "recyclable"       : p["recyclable"],
            "material_type"    : p["material_type"],
            "weight_class"     : p["weight_class"],
            "condition"        : condition,
            "region_risk"      : region_risk,
            "disposal_history" : disposal_history,
            "hazard_score"     : round(score, 2),
        })

df = pd.DataFrame(rows)
csv_path = OUTPUT_DIR / "ewaste_tabular_dataset.csv"
df.to_csv(csv_path, index=False)

print(f"dataset shape   : {df.shape}")
print(f"saved to        : {csv_path}")
print(f"\nhazard score distribution:")
print(df["hazard_score"].describe().round(2))

dataset shape   : (3000, 14)
saved to        : D:\Github Desktop\ewaste_vit_project\models\ann\ewaste_tabular_dataset.csv

hazard score distribution:
count    3000.00
mean       61.98
std        26.85
min         0.00
25%        35.98
50%        67.04
75%        85.24
max       100.00
Name: hazard_score, dtype: float64


## exploratory data analysis

In [4]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("exploratory data analysis — ewaste tabular dataset", fontsize=13, fontweight="bold")

# hazard score distribution
axes[0,0].hist(df["hazard_score"], bins=40, color="#2c7bb6", edgecolor="white", alpha=0.85)
axes[0,0].set_title("hazard score distribution", fontsize=11)
axes[0,0].set_xlabel("hazard score")
axes[0,0].set_ylabel("frequency")
axes[0,0].grid(alpha=0.3)

# per-component mean hazard
comp_means = df.groupby("component")["hazard_score"].mean().sort_values(ascending=True)
axes[0,1].barh(comp_means.index, comp_means.values, color="#4daf4a", alpha=0.85)
axes[0,1].set_title("mean hazard score per component", fontsize=11)
axes[0,1].set_xlabel("mean hazard score")
axes[0,1].tick_params(axis="y", labelsize=7)
axes[0,1].grid(alpha=0.3)

# hazard vs age
axes[0,2].scatter(df["age_years"], df["hazard_score"],
                  alpha=0.3, s=8, color="#d7191c")
z = np.polyfit(df["age_years"], df["hazard_score"], 1)
p = np.poly1d(z)
axes[0,2].plot(sorted(df["age_years"]), p(sorted(df["age_years"])),
               "k--", linewidth=1.5, label="trend")
axes[0,2].set_title("hazard score vs age", fontsize=11)
axes[0,2].set_xlabel("age (years)")
axes[0,2].set_ylabel("hazard score")
axes[0,2].legend()
axes[0,2].grid(alpha=0.3)

# hazard by condition
df.boxplot(column="hazard_score", by="condition", ax=axes[1,0])
axes[1,0].set_title("hazard score by condition", fontsize=11)
axes[1,0].set_xlabel("condition")
axes[1,0].set_ylabel("hazard score")

# hazard by region risk
df.boxplot(column="hazard_score", by="region_risk", ax=axes[1,1])
axes[1,1].set_title("hazard score by region risk", fontsize=11)
axes[1,1].set_xlabel("region risk")
axes[1,1].set_ylabel("hazard score")

# correlation heatmap
num_cols = ["age_years", "weight_kg", "contains_lithium",
            "contains_lead", "contains_mercury", "contains_cadmium",
            "recyclable", "hazard_score"]
corr = df[num_cols].corr()
sns.heatmap(corr, ax=axes[1,2], annot=True, fmt=".2f",
            cmap="RdBu_r", center=0, linewidths=0.5)
axes[1,2].set_title("feature correlation matrix", fontsize=11)
axes[1,2].tick_params(axis="x", rotation=45, labelsize=8)
axes[1,2].tick_params(axis="y", rotation=0, labelsize=8)

plt.suptitle("")
plt.tight_layout()
plt.savefig(GRAPHS_DIR / "eda.png", dpi=150, bbox_inches="tight")
plt.close()
print("eda plots saved")

eda plots saved


## preprocessing

In [5]:
# encode categorical features
le_component = LabelEncoder()
le_material  = LabelEncoder()
le_wclass    = LabelEncoder()
le_condition = LabelEncoder()
le_region    = LabelEncoder()
le_disposal  = LabelEncoder()

df["component_enc"]  = le_component.fit_transform(df["component"])
df["material_enc"]   = le_material.fit_transform(df["material_type"])
df["wclass_enc"]     = le_wclass.fit_transform(df["weight_class"])
df["condition_enc"]  = le_condition.fit_transform(df["condition"])
df["region_enc"]     = le_region.fit_transform(df["region_risk"])
df["disposal_enc"]   = le_disposal.fit_transform(df["disposal_history"])

FEATURES = [
    "component_enc", "age_years", "weight_kg",
    "contains_lithium", "contains_lead", "contains_mercury", "contains_cadmium",
    "recyclable", "material_enc", "wclass_enc",
    "condition_enc", "region_enc", "disposal_enc"
]
TARGET = "hazard_score"

X = df[FEATURES].values.astype(np.float32)
y = df[TARGET].values.astype(np.float32)

# stratified split on hazard class
y_class = pd.cut(y, bins=[0, 40, 70, 100],
                 labels=["LOW", "MEDIUM", "HIGH"]).astype(str)

X_train, X_temp, y_train, y_temp, yc_train, yc_temp = train_test_split(
    X, y, y_class, test_size=0.30, random_state=42, stratify=y_class
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

# scale features
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

print(f"train : {len(X_train)}")
print(f"val   : {len(X_val)}")
print(f"test  : {len(X_test)}")
print(f"features: {FEATURES}")

train : 2100
val   : 450
test  : 450
features: ['component_enc', 'age_years', 'weight_kg', 'contains_lithium', 'contains_lead', 'contains_mercury', 'contains_cadmium', 'recyclable', 'material_enc', 'wclass_enc', 'condition_enc', 'region_enc', 'disposal_enc']


## pytorch dataset

In [6]:
class EWasteDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_ds = EWasteDataset(X_train, y_train)
val_ds   = EWasteDataset(X_val,   y_val)
test_ds  = EWasteDataset(X_test,  y_test)

train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=CONFIG["batch_size"], shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=CONFIG["batch_size"], shuffle=False)
print("datasets ready")

datasets ready


## ann architecture

In [7]:
class HazardANN(nn.Module):
    """
    feedforward network for e-waste hazard score regression.
    architecture: 13 -> 256 -> 128 -> 64 -> 32 -> 1
    uses batch normalization and residual-style skip connection
    for stable training on tabular data.
    """

    def __init__(self, input_dim, dropout_rate=0.3):
        super().__init__()

        self.input_bn = nn.BatchNorm1d(input_dim)

        self.block1 = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate),
        )

        self.block2 = nn.Sequential(
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate * 0.8),
        )

        self.block3 = nn.Sequential(
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate * 0.5),
        )

        self.block4 = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(inplace=True),
        )

        self.head = nn.Linear(32, 1)

        # weight initialization
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.input_bn(x)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        return self.head(x)


model = HazardANN(
    input_dim    = len(FEATURES),
    dropout_rate = CONFIG["dropout_rate"]
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"\ntotal parameters: {total_params:,}")

HazardANN(
  (input_bn): BatchNorm1d(13, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (block1): Sequential(
    (0): Linear(in_features=13, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Dropout(p=0.3, inplace=False)
  )
  (block2): Sequential(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Dropout(p=0.24, inplace=False)
  )
  (block3): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Dropout(p=0.15, inplace=False)
  )
  (block4): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU(inplace=True)
  )
  (head): Linear(in_features=32, out_fe

## training with early stopping

In [8]:
criterion = nn.HuberLoss(delta=5.0)   # robust to outliers vs mse
optimizer = optim.AdamW(
    model.parameters(),
    lr=CONFIG["lr"],
    weight_decay=CONFIG["weight_decay"]
)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", patience=10, factor=0.5, min_lr=1e-6
)

history      = {"train_loss": [], "val_loss": [], "lr": []}
best_val_loss = float("inf")
best_weights  = None
patience_ctr  = 0
best_path     = OUTPUT_DIR / "ann_best.pth"

print(f"training ann for up to {CONFIG['num_epochs']} epochs")
print("-" * 55)

for epoch in range(1, CONFIG["num_epochs"] + 1):
    # train
    model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad(set_to_none=True)
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item() * len(xb)
    train_loss /= len(train_ds)

    # validate
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred     = model(xb)
            val_loss += criterion(pred, yb).item() * len(xb)
    val_loss /= len(val_ds)

    current_lr = optimizer.param_groups[0]["lr"]
    scheduler.step(val_loss)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["lr"].append(current_lr)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_weights  = {k: v.clone() for k, v in model.state_dict().items()}
        torch.save(model.state_dict(), best_path)
        patience_ctr = 0
    else:
        patience_ctr += 1
        if patience_ctr >= CONFIG["patience"]:
            print(f"  early stopping at epoch {epoch}")
            break

    if epoch % 10 == 0:
        print(f"  epoch {epoch:03d} | train {train_loss:.4f} | val {val_loss:.4f} | lr {current_lr:.2e}")

model.load_state_dict(best_weights)
print(f"\nbest val loss: {best_val_loss:.4f}")

training ann for up to 200 epochs
-------------------------------------------------------
  epoch 010 | train 31.0703 | val 18.2146 | lr 1.00e-03
  epoch 020 | train 25.0619 | val 15.4577 | lr 1.00e-03
  epoch 030 | train 22.8921 | val 12.9402 | lr 1.00e-03
  epoch 040 | train 21.6701 | val 12.0834 | lr 1.00e-03
  epoch 050 | train 20.8286 | val 11.2736 | lr 5.00e-04
  epoch 060 | train 20.5122 | val 11.5564 | lr 5.00e-04
  epoch 070 | train 19.7230 | val 11.0495 | lr 2.50e-04
  epoch 080 | train 19.1635 | val 11.3222 | lr 1.25e-04
  epoch 090 | train 20.5848 | val 10.9834 | lr 1.25e-04
  epoch 100 | train 19.8247 | val 10.9689 | lr 1.25e-04
  epoch 110 | train 18.7261 | val 11.0529 | lr 6.25e-05
  early stopping at epoch 120

best val loss: 10.6686


## training curves

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("ann training curves", fontsize=13, fontweight="bold")

epochs = range(1, len(history["train_loss"]) + 1)

axes[0].plot(epochs, history["train_loss"], label="train", color="#2c7bb6", linewidth=1.5)
axes[0].plot(epochs, history["val_loss"],   label="val",   color="#d7191c", linewidth=1.5)
axes[0].set_title("huber loss", fontsize=11)
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs, history["lr"], color="#4daf4a", linewidth=1.5)
axes[1].set_title("learning rate schedule", fontsize=11)
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("learning rate")
axes[1].set_yscale("log")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(GRAPHS_DIR / "ann_training_curves.png", dpi=150, bbox_inches="tight")
plt.close()
print("training curves saved")

training curves saved


## test evaluation

In [10]:
model.eval()
all_preds, all_targets = [], []

with torch.no_grad():
    for xb, yb in test_loader:
        xb       = xb.to(device)
        pred     = model(xb)
        all_preds.extend(pred.cpu().squeeze().numpy())
        all_targets.extend(yb.squeeze().numpy())

all_preds   = np.array(all_preds)
all_targets = np.array(all_targets)

mae  = mean_absolute_error(all_targets, all_preds)
rmse = np.sqrt(mean_squared_error(all_targets, all_preds))
r2   = r2_score(all_targets, all_preds)
mape = np.mean(np.abs((all_targets - all_preds) / (all_targets + 1e-8))) * 100

print("regression metrics:")
print(f"  mae  : {mae:.4f}  (mean error in hazard score points)")
print(f"  rmse : {rmse:.4f}")
print(f"  r2   : {r2:.4f}  (1.0 = perfect fit)")
print(f"  mape : {mape:.2f}%")

regression metrics:
  mae  : 3.7365  (mean error in hazard score points)
  rmse : 4.7489
  r2   : 0.9685  (1.0 = perfect fit)
  mape : 138719664.00%


## predicted vs actual — regression plot

In [11]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("ann hazard score prediction — test set", fontsize=13, fontweight="bold")

# scatter: predicted vs actual
axes[0].scatter(all_targets, all_preds, alpha=0.4, s=15, color="#2c7bb6", label="predictions")
min_val, max_val = min(all_targets.min(), all_preds.min()), max(all_targets.max(), all_preds.max())
axes[0].plot([min_val, max_val], [min_val, max_val], "r--", linewidth=1.5, label="perfect prediction")
axes[0].set_title(f"predicted vs actual  (r2={r2:.4f})", fontsize=11)
axes[0].set_xlabel("actual hazard score")
axes[0].set_ylabel("predicted hazard score")
axes[0].legend()
axes[0].grid(alpha=0.3)

# residuals
residuals = all_preds - all_targets
axes[1].scatter(all_targets, residuals, alpha=0.4, s=15, color="#4daf4a")
axes[1].axhline(y=0, color="red", linestyle="--", linewidth=1.5)
axes[1].axhline(y=mae,  color="orange", linestyle=":", linewidth=1.2, label=f"+mae={mae:.2f}")
axes[1].axhline(y=-mae, color="orange", linestyle=":", linewidth=1.2, label=f"-mae={mae:.2f}")
axes[1].set_title("residuals vs actual", fontsize=11)
axes[1].set_xlabel("actual hazard score")
axes[1].set_ylabel("residual (predicted - actual)")
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(GRAPHS_DIR / "regression_evaluation.png", dpi=150, bbox_inches="tight")
plt.close()
print("regression plots saved")

regression plots saved


## hazard class accuracy — mapping score to class

In [12]:
def to_hazard_class(scores):
    classes = []
    for s in scores:
        if s >= 70:
            classes.append("HIGH")
        elif s >= 40:
            classes.append("MEDIUM")
        else:
            classes.append("LOW")
    return np.array(classes)


pred_classes   = to_hazard_class(all_preds)
target_classes = to_hazard_class(all_targets)

cls_acc = accuracy_score(target_classes, pred_classes)
print(f"hazard class accuracy: {cls_acc:.4f}\n")
print(classification_report(target_classes, pred_classes,
                             target_names=["HIGH", "MEDIUM", "LOW"],
                             digits=4))

# confusion matrix
cm = confusion_matrix(target_classes, pred_classes,
                      labels=["HIGH", "MEDIUM", "LOW"])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("hazard class evaluation", fontsize=13, fontweight="bold")

# normalized confusion matrix
cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt=".3f", cmap="Blues",
            xticklabels=["HIGH", "MEDIUM", "LOW"],
            yticklabels=["HIGH", "MEDIUM", "LOW"],
            linewidths=0.5, ax=axes[0])
axes[0].set_title(f"confusion matrix (normalized)\naccuracy={cls_acc:.4f}", fontsize=11)
axes[0].set_ylabel("true class")
axes[0].set_xlabel("predicted class")

# per-class f1
from sklearn.metrics import f1_score as f1
f1_scores = f1(target_classes, pred_classes, average=None, labels=["HIGH", "MEDIUM", "LOW"])
colors    = ["#d7191c", "#fdae61", "#1a9641"]
axes[1].bar(["HIGH", "MEDIUM", "LOW"], f1_scores, color=colors, alpha=0.85, edgecolor="white")
axes[1].axhline(y=0.85, color="gray", linestyle="--", alpha=0.7, label="target f1=0.85")
axes[1].set_title("per-class f1 score", fontsize=11)
axes[1].set_ylabel("f1 score")
axes[1].set_ylim(0, 1.05)
axes[1].legend()
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(GRAPHS_DIR / "hazard_class_evaluation.png", dpi=150, bbox_inches="tight")
plt.close()
print("class evaluation plots saved")

hazard class accuracy: 0.8911

              precision    recall  f1-score   support

        HIGH     0.9340    0.9293    0.9316       198
      MEDIUM     0.8675    0.9850    0.9225       133
         LOW     0.8431    0.7227    0.7783       119

    accuracy                         0.8911       450
   macro avg     0.8816    0.8790    0.8775       450
weighted avg     0.8903    0.8911    0.8884       450

class evaluation plots saved


## feature importance — gradient sensitivity analysis

In [16]:
model.eval()
X_tensor = torch.tensor(X_test, dtype=torch.float32, requires_grad=True).to(device)
preds     = model(X_tensor)
grads = torch.autograd.grad(outputs=preds.sum(), inputs=X_tensor, create_graph=False)[0]
importance = grads.abs().mean(dim=0).cpu().detach().numpy()
importance = importance / importance.sum()  # normalize to percentages

sorted_idx = np.argsort(importance)[::-1]
feat_names = [FEATURES[i] for i in sorted_idx]
feat_imp   = importance[sorted_idx]

plt.figure(figsize=(12, 6))
bar_colors = ["#d7191c" if v > 0.12 else "#fdae61" if v > 0.07 else "#2c7bb6"
              for v in feat_imp]
plt.bar(feat_names, feat_imp, color=bar_colors, alpha=0.85, edgecolor="white")
plt.title("feature importance — gradient sensitivity analysis", fontsize=12, fontweight="bold")
plt.xlabel("feature")
plt.ylabel("normalized importance")
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(GRAPHS_DIR / "feature_importance.png", dpi=150, bbox_inches="tight")
plt.close()

print("feature importance (top 5):")
for f, v in zip(feat_names[:5], feat_imp[:5]):
    print(f"  {f:<25} : {v:.4f}  ({v*100:.1f}%)")

feature importance (top 5):
  wclass_enc                : 0.1332  (13.3%)
  contains_lead             : 0.1329  (13.3%)
  condition_enc             : 0.1084  (10.8%)
  region_enc                : 0.1072  (10.7%)
  material_enc              : 0.1010  (10.1%)


## save all results

In [17]:
results = {
    "model"               : "HazardANN",
    "architecture"        : "13->256->128->64->32->1",
    "total_parameters"    : sum(p.numel() for p in model.parameters()),
    "features"            : FEATURES,
    "n_samples"           : {"train": len(train_ds), "val": len(val_ds), "test": len(test_ds)},
    "regression_metrics"  : {
        "mae"  : round(float(mae),  4),
        "rmse" : round(float(rmse), 4),
        "r2"   : round(float(r2),   4),
        "mape" : round(float(mape), 2),
    },
    "classification_metrics": {
        "hazard_class_accuracy": round(float(cls_acc), 4),
        "f1_HIGH"   : round(float(f1_scores[0]), 4),
        "f1_MEDIUM" : round(float(f1_scores[1]), 4),
        "f1_LOW"    : round(float(f1_scores[2]), 4),
    },
    "top_features": [
        {"feature": feat_names[i], "importance": round(float(feat_imp[i]), 4)}
        for i in range(5)
    ]
}

with open(OUTPUT_DIR / "ann_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("ann results saved")
print(json.dumps(results, indent=2))

ann results saved
{
  "model": "HazardANN",
  "architecture": "13->256->128->64->32->1",
  "total_parameters": 47771,
  "features": [
    "component_enc",
    "age_years",
    "weight_kg",
    "contains_lithium",
    "contains_lead",
    "contains_mercury",
    "contains_cadmium",
    "recyclable",
    "material_enc",
    "wclass_enc",
    "condition_enc",
    "region_enc",
    "disposal_enc"
  ],
  "n_samples": {
    "train": 2100,
    "val": 450,
    "test": 450
  },
  "regression_metrics": {
    "mae": 3.7365,
    "rmse": 4.7489,
    "r2": 0.9685,
    "mape": 138719664.0
  },
  "classification_metrics": {
    "hazard_class_accuracy": 0.8911,
    "f1_HIGH": 0.9316,
    "f1_MEDIUM": 0.7783,
    "f1_LOW": 0.9225
  },
  "top_features": [
    {
      "feature": "wclass_enc",
      "importance": 0.1332
    },
    {
      "feature": "contains_lead",
      "importance": 0.1329
    },
    {
      "feature": "condition_enc",
      "importance": 0.1084
    },
    {
      "feature": "region_enc

## ann complete
Outputs saved:
- model: `models/ann/ann_best.pth`
- dataset: `models/ann/ewaste_tabular_dataset.csv`
- results: `models/ann/ann_results.json`
- graphs: `models/ann/graphs/`